# CHOP Workshop: LLMs in Data Extraction & AISQL
## Hands-On Exercises: AI Functions + Cortex Intelligence

---

### Before you start — set your role

| You are... | Role | Warehouse | Track |
|-----------|------|----------|-------|
| Data Analyst | `CHOP_SNOW_INTELLIGENCE` | `CHOP_SNOW_INTELLIGENCE_WH` | **SQL** |
| Data Scientist / ML Engineer | `ML_ENGINEER` | `HEALTHCARE_ML_WH` | **Python** |

Run **Cell 1** first — it detects your role, confirms your environment is ready, and shows exactly which cells to run.

---

### Session agenda

| Cell | Exercise | Cortex Function | Analysts (SQL) | Scientists (Python) |
|------|----------|-----------------|:--------------:|:-------------------:|
| 1 | Environment check + routing | — | Run | Run |
| 2-SQL / 2-Py | Extraction patterns | `AI_EXTRACT` | SQL only | Python only |
| 3-SQL / 3-Py | Guardrails & classification | `AI_CLASSIFY` | SQL only | Python only |
| 4-SQL / 4-Py | Quality gates + transformation | `AI_FILTER`, `COMPLETE` | SQL only | Python only |
| 5 | Cortex Code bridge → SI | coco CLI (SE-led) | Watch | Watch |
| — | **Break** | — | — | — |
| 6 | SI call sheet & agent demo | (SE drives) | Watch | Watch |

> **Cell referencing:** SQL cells reference `{{role}}` from Cell 1.  
> Python cells chain results across exercises: `df_extracted` → `df_classified` → `df_filtered`.

In [ ]:
# =============================================================================
# CELL 1: Environment setup + readiness check
# =============================================================================
# Run this first. It confirms role, warehouse, and data access are all working.
# If any check shows FAIL, contact your SE before the exercises start.

from snowflake.snowpark.context import get_active_session
from IPython.display import display, Markdown
import json

session = get_active_session()

role      = session.get_current_role().strip('"')
warehouse = session.get_current_warehouse().strip('"')
print(f"Role:      {role}")
print(f"Warehouse: {warehouse}")
print()

# --- Role-specific config ---
if role == 'CHOP_SNOW_INTELLIGENCE':
    expected_wh  = 'CHOP_SNOW_INTELLIGENCE_WH'
    check_table  = 'SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS'
    participant  = 'Data Analyst'
elif role == 'ML_ENGINEER':
    expected_wh  = 'HEALTHCARE_ML_WH'
    check_table  = 'HEALTHCARE_ML.RAW_DATA.ADMISSIONS'
    participant  = 'Data Scientist'
else:
    print(f"WARNING: unexpected role '{role}'.")
    print("Expected CHOP_SNOW_INTELLIGENCE or ML_ENGINEER. Contact your SE.")
    expected_wh  = ''
    check_table  = ''
    participant  = 'Unknown'

print(f"Participant type: {participant}")
print()

# --- Checks ---
checks = []

# Check 1: Warehouse
wh_ok = warehouse == expected_wh
checks.append(('Warehouse', warehouse, 'PASS' if wh_ok else f'FAIL – expected {expected_wh}'))

# Check 2: Table / view access
if check_table:
    try:
        row_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {check_table}").collect()[0]["CNT"]
        checks.append(('Data access', f'{row_count:,} rows in {check_table.split(".")[-1]}',
                        'PASS' if row_count > 0 else 'FAIL – 0 rows returned'))
    except Exception as e:
        checks.append(('Data access', str(e)[:80], 'FAIL – query error'))

# Check 3: Cortex AI function
try:
    result = session.sql("""
        SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
            'take 1 tablet by mouth daily',
            ['oral','intravenous','subcutaneous']
        )::VARCHAR AS label
    """).collect()[0]["LABEL"]
    checks.append(('Cortex AI_CLASSIFY', result[:60], 'PASS'))
except Exception as e:
    checks.append(('Cortex AI_CLASSIFY', str(e)[:80], 'FAIL – Cortex not enabled for role'))

# --- Print summary ---
print(f"{'CHECK':<22} {'RESULT':<50} {'STATUS'}")
print("-" * 85)
for name, value, status in checks:
    icon = '✓' if status == 'PASS' else '✗'
    print(f"{icon} {name:<20} {value:<50} {status}")

all_pass = all(c[2] == 'PASS' for c in checks)
print()
print("Environment ready — let's go!" if all_pass else ">>> One or more checks failed. Contact your SE. <<<")

# --- Role-based navigation ---
print()
if role == 'CHOP_SNOW_INTELLIGENCE':
    display(Markdown("""
---
### Your track: SQL (Data Analyst)

For each exercise, **run the SQL cell** (labelled `ANALYSTS ONLY`). Skip the Python cell below it.

| Exercise | Your cell | Skip |
|----------|-----------|------|
| 1 — AI_EXTRACT | `ex1_extract_sql` | `ex1_extract_py` |
| 2 — AI_CLASSIFY | `ex2_classify_sql` | `ex2_classify_py` |
| 3 — AI_FILTER + COMPLETE | `ex3_filter_sql` | `ex3_filter_py` |

→ **Scroll down to the Exercise 1 header.**
"""))
elif role == 'ML_ENGINEER':
    display(Markdown("""
---
### Your track: Python (Data Scientist)

For each exercise, **run the Python cell** (labelled `SCIENTISTS ONLY`). Skip the SQL cell above it.

| Exercise | Your cell | Skip |
|----------|-----------|------|
| 1 — AI_EXTRACT | `ex1_extract_py` | `ex1_extract_sql` |
| 2 — AI_CLASSIFY | `ex2_classify_py` | `ex2_classify_sql` |
| 3 — AI_FILTER + COMPLETE | `ex3_filter_py` | `ex3_filter_sql` |

Python cells chain results across exercises: `df_extracted` → `df_classified` → `df_filtered`.  
→ **Scroll down to the Exercise 1 header.**
"""))

---
## Exercise 1 — AI_EXTRACT: Extraction patterns

**Goal:** Extract structured entities from free-text clinical instructions.

### Part A — Concept (shared warmup, no cell to run)

`AI_EXTRACT` pulls named fields from free text. You define the field names — the model finds them:

```sql
SELECT SNOWFLAKE.CORTEX.AI_EXTRACT(
    'Administer methotrexate 25mg subcutaneously once weekly.
     Monitor LFTs monthly. Hold if WBC < 3.0.',
    ['medication_name', 'dose', 'dose_unit', 'route',
     'frequency', 'monitoring_instructions', 'hold_condition']
) AS extracted_entities
```

**Key behaviour:** Only fields the model finds are returned — missing fields are omitted, not null.

Try it mentally: `'take 2 tablets by mouth daily'` → which of the 7 fields above would be returned?

---

### Part B — Your exercise

| Your role | Run this cell | Data source |
|-----------|--------------|-------------|
| `CHOP_SNOW_INTELLIGENCE` | **`ex1_extract_sql`** ↓ (SQL cell) | Real CHOP pharmacy SIG text |
| `ML_ENGINEER` | **`ex1_extract_py`** ↓↓ (Python cell) | ADMISSIONS clinical narratives |

In [ ]:
-- =============================================================================
-- ex1_extract_sql  |  ANALYSTS ONLY (CHOP_SNOW_INTELLIGENCE)
-- Exercise 1: AI_EXTRACT on real CHOP pharmacy SIG text
-- Running as: {{role}}
-- =============================================================================
-- Compare results across rows: does the model always find all 5 fields?
-- Notice when SIG text is vague — which fields go missing?

SELECT
    ADMINISTRATIONDIRECTIONS                              AS raw_sig,
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        ADMINISTRATIONDIRECTIONS,
        ['medication_name', 'dose', 'route', 'frequency', 'duration']
    )                                                     AS structured_entities
FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
LIMIT 5

In [ ]:
# =============================================================================
# ex1_extract_py  |  SCIENTISTS ONLY (ML_ENGINEER)
# Exercise 1: AI_EXTRACT on ADMISSIONS clinical narratives
# Cell reference: stores df_extracted — used by ex2_classify_py
# =============================================================================
assert role == 'ML_ENGINEER', \
    f"This cell is for ML_ENGINEER. You are '{role}' — skip to ex1_extract_sql above."

print("=== AI_EXTRACT: building clinical narratives → structured extraction ===")
print("Your data is structured, but AI_EXTRACT works on text — construct it first.\n")

df_extracted = session.sql("""
    SELECT
        ADMISSION_ID,
        'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
        ', discharged to ' || DISCHARGE_DISPOSITION ||
        ', LOS ' || LENGTH_OF_STAY || ' days, ' ||
        NUM_DIAGNOSES || ' diagnoses, ' ||
        NUM_PROCEDURES || ' procedures.'              AS clinical_narrative,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
            ', discharged to ' || DISCHARGE_DISPOSITION ||
            ', LOS ' || LENGTH_OF_STAY || ' days, ' ||
            NUM_DIAGNOSES || ' diagnoses, ' ||
            NUM_PROCEDURES || ' procedures.',
            ['primary_condition', 'discharge_setting',
             'complexity_indicators', 'readmission_risk_factors']
        )::VARCHAR                                    AS extracted_summary
    FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
    LIMIT 5
""").to_pandas()

for _, row in df_extracted.iterrows():
    print(f"Narrative: {row['CLINICAL_NARRATIVE']}")
    print(f"Extracted: {row['EXTRACTED_SUMMARY'][:120]}")
    print()

print(f"df_extracted: {len(df_extracted)} rows stored — passed to ex2_classify_py via cell reference.")

In [ ]:
# =============================================================================
# compare-ex1  |  BOTH ROLES — Compare tracks
# =============================================================================
# Re-queries both data sources (LIMIT 3) so each audience can see what
# the other track extracted. Gracefully skips any table you don't have access to.

print("=== Compare tracks: Exercise 1 — AI_EXTRACT ===")
print("Same function. Completely different input data. Same structured output shape.\n")

print("--- SQL track (CHOP_SNOW_INTELLIGENCE): pharmacy SIG extraction ---")
try:
    sql_df = session.sql("""
        SELECT
            ADMINISTRATIONDIRECTIONS                              AS raw_sig,
            SNOWFLAKE.CORTEX.AI_EXTRACT(
                ADMINISTRATIONDIRECTIONS,
                ['medication_name', 'dose', 'route', 'frequency', 'duration']
            )::VARCHAR                                            AS structured_entities
        FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
        WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
        LIMIT 3
    """).to_pandas()
    display(sql_df)
except Exception as e:
    print(f"(skipped — {e})")

print()
print("--- Python track (ML_ENGINEER): ADMISSIONS narrative extraction ---")
try:
    display(df_extracted[['CLINICAL_NARRATIVE', 'EXTRACTED_SUMMARY']].head(3))
except NameError:
    try:
        py_df = session.sql("""
            SELECT
                'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
                ', discharged to ' || DISCHARGE_DISPOSITION ||
                ', LOS ' || LENGTH_OF_STAY || ' days'             AS clinical_narrative,
                SNOWFLAKE.CORTEX.AI_EXTRACT(
                    'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
                    ', discharged to ' || DISCHARGE_DISPOSITION ||
                    ', LOS ' || LENGTH_OF_STAY || ' days',
                    ['primary_condition', 'discharge_setting',
                     'complexity_indicators', 'readmission_risk_factors']
                )::VARCHAR                                         AS extracted_summary
            FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
            LIMIT 3
        """).to_pandas()
        display(py_df)
    except Exception as e:
        print(f"(skipped — {e})")

---
## Exercise 2 — AI_CLASSIFY: Guardrails & quality evaluation

**Goal:** Use AI classification as a guardrail — catch bad or inconsistent values before they reach downstream tables or models.

### Part A — Concept (shared warmup, no cell to run)

`AI_CLASSIFY` assigns a label from a fixed list you define:

```sql
SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
    'infuse 500ml normal saline over 4 hours through peripheral IV line',
    ['oral', 'intravenous', 'intramuscular', 'subcutaneous', 'topical', 'inhaled']
)
-- Returns: {"label": "intravenous", "score": 0.97}
```

**Key insight:** You control the label vocabulary. The model picks the closest match — even for messy, informal text.

---

### Part B — Your exercise

| Your role | Run this cell | What you're comparing |
|-----------|--------------|----------------------|
| `CHOP_SNOW_INTELLIGENCE` | **`ex2_classify_sql`** ↓ (SQL cell) | AI route label vs stored `ADMINROUTE` — find mismatches |
| `ML_ENGINEER` | **`ex2_classify_py`** ↓↓ (Python cell) | AI risk tier for the admissions from `df_extracted` |

In [ ]:
-- =============================================================================
-- ex2_classify_sql  |  ANALYSTS ONLY (CHOP_SNOW_INTELLIGENCE)
-- Exercise 2: AI_CLASSIFY — compare AI route label vs stored ADMINROUTE
-- Running as: {{role}}
-- Cell reference: same source table as ex1_extract_sql
-- =============================================================================
-- Where AI and stored values disagree = your data quality signal.
-- Discussion: should you trust the AI label or the stored value? When?

SELECT
    ADMINISTRATIONDIRECTIONS                              AS raw_sig,
    ADMINROUTE                                            AS stored_route,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        ADMINISTRATIONDIRECTIONS,
        ['oral','intravenous','intramuscular','subcutaneous',
         'topical','inhaled','other']
    )::VARCHAR                                            AS ai_classification
FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
LIMIT 5

In [ ]:
# =============================================================================
# ex2_classify_py  |  SCIENTISTS ONLY (ML_ENGINEER)
# Exercise 2: AI_CLASSIFY — discharge disposition → readmission risk tier
# Cell reference: reads df_extracted (ADMISSION_IDs) from ex1_extract_py
#                 stores df_classified — used by ex3_filter_py
# =============================================================================
assert role == 'ML_ENGINEER', \
    f"This cell is for ML_ENGINEER. You are '{role}' — skip to ex2_classify_sql above."

print("=== AI_CLASSIFY: discharge disposition → readmission risk tier ===")
print("Cell reference: scoped to admission IDs from df_extracted\n")

# Cell reference: use admission IDs already extracted in ex1_extract_py
admission_ids = df_extracted['ADMISSION_ID'].tolist()
ids_str = ', '.join(f"'{aid}'" for aid in admission_ids)

df_classified = session.sql(f"""
    SELECT
        ADMISSION_ID,
        DISCHARGE_DISPOSITION,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            'Patient discharged to: ' || DISCHARGE_DISPOSITION,
            ['high_readmission_risk',
             'medium_readmission_risk',
             'low_readmission_risk']
        )::VARCHAR                      AS ai_risk_tier,
        READMITTED_30D                  AS actual_outcome
    FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
    WHERE ADMISSION_ID IN ({ids_str})
""").to_pandas()

for _, row in df_classified.iterrows():
    tier    = json.loads(row['AI_RISK_TIER']).get('label', '?').replace('_readmission_risk', '')
    outcome = 'readmitted' if row['ACTUAL_OUTCOME'] == 1 else 'not readmitted'
    print(f"{row['ADMISSION_ID']} | {str(row['DISCHARGE_DISPOSITION']):<12} → AI risk: {tier:<8} | actual: {outcome}")

print()
print(f"df_classified: {len(df_classified)} rows stored — passed to ex3_filter_py via cell reference.")
print("Discussion: Does AI risk tier align with your DISPOSITION_RISK_SCORE engineered feature?")
print("LLM-based features vs hand-engineered features — when does each approach win?")

In [ ]:
# =============================================================================
# compare-ex2  |  BOTH ROLES — Compare tracks
# =============================================================================
# Re-queries both data sources (LIMIT 3) so each audience can see what
# the other track classified. Gracefully skips any table you don't have access to.

print("=== Compare tracks: Exercise 2 — AI_CLASSIFY ===")
print("Same function, same label mechanics — one enforces data quality, one builds ML features.\n")

print("--- SQL track (CHOP_SNOW_INTELLIGENCE): route classification vs stored ADMINROUTE ---")
try:
    sql_df = session.sql("""
        SELECT
            ADMINISTRATIONDIRECTIONS                              AS raw_sig,
            ADMINROUTE                                            AS stored_route,
            SNOWFLAKE.CORTEX.AI_CLASSIFY(
                ADMINISTRATIONDIRECTIONS,
                ['oral','intravenous','intramuscular','subcutaneous',
                 'topical','inhaled','other']
            )::VARCHAR                                            AS ai_classification
        FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
        WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
        LIMIT 3
    """).to_pandas()
    display(sql_df)
except Exception as e:
    print(f"(skipped — {e})")

print()
print("--- Python track (ML_ENGINEER): discharge disposition → readmission risk tier ---")
try:
    display(df_classified[['ADMISSION_ID', 'DISCHARGE_DISPOSITION', 'AI_RISK_TIER', 'ACTUAL_OUTCOME']].head(3))
except NameError:
    try:
        py_df = session.sql("""
            SELECT
                ADMISSION_ID,
                DISCHARGE_DISPOSITION,
                SNOWFLAKE.CORTEX.AI_CLASSIFY(
                    'Patient discharged to: ' || DISCHARGE_DISPOSITION,
                    ['high_readmission_risk',
                     'medium_readmission_risk',
                     'low_readmission_risk']
                )::VARCHAR                      AS ai_risk_tier,
                READMITTED_30D                  AS actual_outcome
            FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
            LIMIT 3
        """).to_pandas()
        display(py_df)
    except Exception as e:
        print(f"(skipped — {e})")

---
## Exercise 3 — AI_FILTER + COMPLETE: Quality gates & transformation

**Goal:** Gate low-quality records with `AI_FILTER` before they enter your pipeline, then use `COMPLETE` to standardize the records that pass.

### Part A — Concept (shared warmup, no cell to run)

**`AI_FILTER`** returns `TRUE/FALSE` — use it directly in a `WHERE` clause:

```sql
WHERE SNOWFLAKE.CORTEX.AI_FILTER(
    sig_text,
    'A complete prescription direction specifying dose, frequency, and route'
) = TRUE
```

**`COMPLETE`** (free-form LLM) standardizes informal text. Chain the two together:
1. `AI_FILTER` in `WHERE` → keep only complete records
2. `COMPLETE` on the survivors → standardize them

**Key insight:** AI_FILTER + COMPLETE + AI_EXTRACT in a single SQL query = structured, clean data from messy clinical notes.

---

### Part B — Your exercise

| Your role | Run this cell | What you're doing |
|-----------|--------------|-------------------|
| `CHOP_SNOW_INTELLIGENCE` | **`ex3_filter_sql`** ↓ (SQL cell) | Filter complete pharmacy SIGs → standardize survivors |
| `ML_ENGINEER` | **`ex3_filter_py`** ↓↓ (Python cell) | Filter high-complexity admissions → generate risk narratives |

In [ ]:
-- =============================================================================
-- ex3_filter_sql  |  ANALYSTS ONLY (CHOP_SNOW_INTELLIGENCE)
-- Exercise 3: AI_FILTER + COMPLETE chained on pharmacy SIG text
-- Running as: {{role}}
-- =============================================================================
-- Step 1 (AI_FILTER WHERE): only SIGs with dose + frequency + route pass through.
-- Step 2 (COMPLETE): standardize the survivors into a clean clinical format.
-- Extension: add AI_EXTRACT on standardized_sig for even cleaner structured output.

WITH filtered_sigs AS (
    SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
    FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
    WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
      AND SNOWFLAKE.CORTEX.AI_FILTER(
              'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
              || ADMINISTRATIONDIRECTIONS
          ) = TRUE
    LIMIT 3
)
SELECT
    raw_sig,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'Standardize this prescription SIG into the format: '
        || '[Drug] [dose] [route] [frequency] [duration] [monitoring]. '
        || 'Be concise. SIG: "' || raw_sig || '"'
    ) AS standardized_sig
FROM filtered_sigs

In [ ]:
# =============================================================================
# ex3_filter_py  |  SCIENTISTS ONLY (ML_ENGINEER)
# Exercise 3: AI_FILTER + COMPLETE on ADMISSIONS
# Cell reference: reads df_classified (ADMISSION_IDs) from ex2_classify_py
# =============================================================================
assert role == 'ML_ENGINEER', \
    f"This cell is for ML_ENGINEER. You are '{role}' — skip to ex3_filter_sql above."

print("=== AI_FILTER + COMPLETE: filter high-complexity admissions → risk narratives ===")
print("Cell reference: scoped to admission IDs from df_classified\n")

# Cell reference: use admission IDs already classified in ex2_classify_py
admission_ids = df_classified['ADMISSION_ID'].tolist()
ids_str = ', '.join(f"'{aid}'" for aid in admission_ids)

df_filtered = session.sql(f"""
    WITH source AS (
        SELECT
            ADMISSION_ID,
            PRIMARY_DIAGNOSIS,
            DISCHARGE_DISPOSITION,
            LENGTH_OF_STAY,
            NUM_DIAGNOSES,
            NUM_PROCEDURES,
            'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
            ', LOS ' || LENGTH_OF_STAY || ' days, ' ||
            NUM_DIAGNOSES || ' diagnoses, ' ||
            NUM_PROCEDURES || ' procedures, discharged to ' ||
            DISCHARGE_DISPOSITION                       AS admission_summary
        FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
        WHERE ADMISSION_ID IN ({ids_str})
    )
    SELECT
        ADMISSION_ID,
        admission_summary,
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-70b',
            'You are a clinical risk analyst. Write a concise readmission risk narrative '
            || '(2 sentences max) highlighting key risk factors for: "'
            || admission_summary || '"'
        ) AS risk_narrative
    FROM source
    WHERE SNOWFLAKE.CORTEX.AI_FILTER(
        'Is this a complex or high-risk admission based on multiple diagnoses, long stay, or high-risk discharge setting? Case: '
        || admission_summary
    ) = TRUE
""").to_pandas()

print(f"Records passing quality gate: {len(df_filtered)} of {len(df_classified)}\n")
for _, row in df_filtered.iterrows():
    print(f"ID:        {row['ADMISSION_ID']}")
    print(f"Summary:   {row['ADMISSION_SUMMARY'][:80]}")
    print(f"Narrative: {row['RISK_NARRATIVE'].strip()[:200]}")
    print()

print("Extension: add AI_EXTRACT on risk_narrative → structured ML features from generated text.")

In [ ]:
# =============================================================================
# compare-ex3  |  BOTH ROLES — Compare tracks
# =============================================================================
# Re-queries both data sources (LIMIT 3) so each audience can see what
# the other track filtered and standardized. Gracefully skips tables you don't have access to.

print("=== Compare tracks: Exercise 3 — AI_FILTER + COMPLETE ===")
print("Same pipeline pattern — quality gate then standardize. Domain differs, structure is identical.\n")

print("--- SQL track (CHOP_SNOW_INTELLIGENCE): filtered + standardized pharmacy SIGs ---")
try:
    sql_df = session.sql("""
        WITH filtered_sigs AS (
            SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
            FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
            WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
              AND SNOWFLAKE.CORTEX.AI_FILTER(
                      'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
                      || ADMINISTRATIONDIRECTIONS
                  ) = TRUE
            LIMIT 3
        )
        SELECT
            raw_sig,
            SNOWFLAKE.CORTEX.COMPLETE(
                'llama3.1-70b',
                'Standardize this SIG into: [Drug] [dose] [route] [frequency]. Be concise. SIG: "' || raw_sig || '"'
            ) AS standardized_sig
        FROM filtered_sigs
    """).to_pandas()
    display(sql_df)
except Exception as e:
    print(f"(skipped — {e})")

print()
print("--- Python track (ML_ENGINEER): high-complexity admissions + risk narratives ---")
try:
    display(df_filtered[['ADMISSION_ID', 'ADMISSION_SUMMARY', 'RISK_NARRATIVE']].head(3))
except NameError:
    try:
        py_df = session.sql("""
            WITH source AS (
                SELECT
                    ADMISSION_ID,
                    'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
                    ', LOS ' || LENGTH_OF_STAY || ' days, discharged to ' ||
                    DISCHARGE_DISPOSITION                       AS admission_summary
                FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
            )
            SELECT
                ADMISSION_ID,
                admission_summary,
                SNOWFLAKE.CORTEX.COMPLETE(
                    'llama3.1-70b',
                    'Write a 1-sentence readmission risk note for: "' || admission_summary || '"'
                ) AS risk_narrative
            FROM source
            WHERE SNOWFLAKE.CORTEX.AI_FILTER(
                'Is this a complex or high-risk admission? Case: '
                || admission_summary
            ) = TRUE
            LIMIT 3
        """).to_pandas()
        display(py_df)
    except Exception as e:
        print(f"(skipped — {e})")

---
## Bonus — Full pipeline

**Goal:** Chain all three functions (`AI_FILTER` → `AI_EXTRACT` → `AI_CLASSIFY`) in a single pass — the production pattern for clinical data enrichment.

| Your role | Run this cell |
|-----------|---------------|
| `CHOP_SNOW_INTELLIGENCE` | **`bonus-sql`** ↓ (SQL cell) |
| `ML_ENGINEER` | **`bonus-py`** ↓↓ (Python cell) |

In [ ]:
-- =============================================================================
-- bonus-sql  |  ANALYSTS — Full pipeline: AI_FILTER → AI_EXTRACT → AI_CLASSIFY
-- Running as: {{role}}
-- =============================================================================
-- Production pattern: one CTE chain, one table scan, three AI functions.
-- Step 1: AI_FILTER keeps only complete SIGs (dose + route + frequency).
-- Step 2: AI_EXTRACT pulls structured entities from the survivors.
-- Step 3: AI_CLASSIFY assigns a validated route label for downstream use.

WITH quality_checked AS (
    SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
    FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
    WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
      AND SNOWFLAKE.CORTEX.AI_FILTER(
              'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
              || ADMINISTRATIONDIRECTIONS
          ) = TRUE
    LIMIT 5
),
extracted AS (
    SELECT
        raw_sig,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            raw_sig,
            ['medication_name', 'dose', 'route', 'frequency']
        )::VARCHAR AS entities
    FROM quality_checked
)
SELECT
    raw_sig,
    entities,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        raw_sig,
        ['oral', 'intravenous', 'intramuscular', 'subcutaneous', 'topical', 'inhaled', 'other']
    )::VARCHAR AS validated_route
FROM extracted

In [ ]:
# =============================================================================
# bonus-py  |  SCIENTISTS — Full pipeline: merge df_extracted + df_classified
# Cell references: df_extracted (ex1_extract_py) + df_classified (ex2_classify_py)
# =============================================================================
# Assembles the complete ML feature row: extracted entities + risk tier + outcome.
# This is the feature table you'd INSERT INTO FEATURE_STORE or pass to model.predict().

print("=== BONUS: unified ML feature table from cell references ===")
print("Merging df_extracted + df_classified on ADMISSION_ID\n")

try:
    df_pipeline = df_extracted[['ADMISSION_ID', 'CLINICAL_NARRATIVE', 'EXTRACTED_SUMMARY']].merge(
        df_classified[['ADMISSION_ID', 'DISCHARGE_DISPOSITION', 'AI_RISK_TIER', 'ACTUAL_OUTCOME']],
        on='ADMISSION_ID',
        how='inner'
    )

    for _, row in df_pipeline.iterrows():
        tier    = json.loads(row['AI_RISK_TIER']).get('label', '?').replace('_readmission_risk', '')
        outcome = 'readmitted' if row['ACTUAL_OUTCOME'] == 1 else 'not readmitted'
        print(f"ID:        {row['ADMISSION_ID']}")
        print(f"Narrative: {row['CLINICAL_NARRATIVE'][:80]}")
        print(f"Extracted: {row['EXTRACTED_SUMMARY'][:80]}")
        print(f"Risk tier: {tier:<8} | Actual: {outcome}")
        print()

    print(f"Pipeline: {len(df_pipeline)} rows with extraction + classification features.")
    print("Next step: INSERT INTO HEALTHCARE_ML.FEATURE_STORE or pass to model.predict().")
except NameError as e:
    print(f"Missing cell reference: {e}")
    print("Run ex1_extract_py and ex2_classify_py first.")

# What we just built — and what comes next

---

In the last 25 minutes you've used four Cortex AI functions:

| Function | What it does | Workshop use |
|----------|-------------|-------------|
| `AI_EXTRACT` | Pull structured fields from free text | Extract medication entities from SIG |
| `AI_CLASSIFY` | Assign labels from a controlled list | Route guardrails, readmission risk |
| `AI_FILTER` | Boolean quality gate | Keep only complete, usable records |
| `COMPLETE` | Free-form transformation | Standardize informal notes |

These work inline in SQL — any `SELECT` statement, any table.

---

## The gap they don't close

Your analysts still need to know the right SQL, the right table names, the right filters.  
Every new question = new query = someone from the data team.

**What if analysts could ask in plain English, and the system figured out the SQL?**

That's what a Cortex Agent does:  
- A **semantic view** teaches it the data shape and business meaning  
- A **search service** lets it fuzzy-search free-text content  
- A **generic function** (like `EXTRACT_PRESCRIPTION_ENTITIES`) is one of its tools  

---

## Live: Cortex Code generates the agent spec

Your SE will now type this into the **Cortex Code CLI** (`coco`):

```
I have these objects in SI_CHOP.CHOP_SNOW_INTELLIGENCE:

- PRESCRIPTION_ORDERS_SV: semantic view on pharmacy prescriptions
  (facts: script_number, costs, quantities; dimensions: drug, route, SIG)
- MEDICATION_ORDERS_SV: semantic view on Epic medication orders
- DRUG_CATALOG_SEARCH: Cortex Search on drug catalog (name, NDC, category)
- PRESCRIPTION_DIRECTIONS_SEARCH: Cortex Search on SIG free text
- EXTRACT_PRESCRIPTION_ENTITIES(VARCHAR): UDF — calls AI_EXTRACT on SIG text
- GENERATE_STREAMLIT_APP(VARCHAR): procedure — generates a Streamlit dashboard

Generate the SQL to CREATE a Cortex Agent that uses all 6 as tools
and can answer pharmacy questions in natural language.
```

Watch coco generate the `CREATE AGENT` specification.  
**That specification is essentially what is already deployed as `CHOP_Pharmacy_Intelligence_Agent`.**

---

**→ Take a 10-minute break, then we demo the live agent.**

# Snowflake Intelligence — Agent Demo Call Sheet

---

## Accessing the agent

1. Open Snowsight  
2. Left navigation → **Agents**  
3. Click **CHOP_Pharmacy_Intelligence_Agent**  
4. Start typing your question in the chat

---

## 5 questions to try — pick one, type it in

Each question exercises a different agent tool:

| # | Question | Tool used |
|---|----------|----------|
| 1 | *"What are the top 10 most prescribed drugs by script count?"* | Cortex Analyst → PRESCRIPTION_ORDERS_SV |
| 2 | *"Show me all IV-route prescriptions from the last 30 days"* | Cortex Analyst → PRESCRIPTION_ORDERS_SV |
| 3 | *"Find drugs related to amoxicillin in the drug catalog"* | Cortex Search → DRUG_CATALOG_SEARCH |
| 4 | *"Extract the medication name and dose from: 'Give 25mg MTX SQ weekly, hold if WBC < 3'"* | Generic function → EXTRACT_PRESCRIPTION_ENTITIES |
| 5 | *"What therapeutic classes have the most medication orders?"* | Cortex Analyst → MEDICATION_ORDERS_SV |

---

## Architecture — how the agent works

```
User question (natural language)
        │
        ▼
   Cortex Agent (orchestrator LLM)
        │
   ┌────┴────────────────────────────────┐
   │                                     │
   ▼                                     ▼
Cortex Analyst           Cortex Search        Generic Function
(text-to-SQL on          (semantic fuzzy       (EXTRACT_PRESCRIPTION_ENTITIES
semantic views)          lookup on SIG text    wraps AI_EXTRACT — same
                         and drug catalog)     function you used in Cell 2)
   │                                     │
   ▼                                     ▼
PRESCRIPTION_ORDERS_SV       DRUG_CATALOG_SEARCH
MEDICATION_ORDERS_SV         PRESCRIPTION_DIRECTIONS_SEARCH
   │
   ▼
V_PHARMACY_ALLMEDICALSCRIPTS  (the view with 12-month filter + 50K cap)
   │
   ▼
PROD.LAKE_HDMS.DS_PHARMACY_ALLMEDICALSCRIPTS  (source of truth)
```

---

## For data scientists — bridge to your ML work

The agent extracts entities from SIG text automatically.  
Those same entities (medication name, dose, route) could be features in a readmission risk model.  

**Discussion:** How would you build a readmission risk agent using the HEALTHCARE_ML objects you have?
- Semantic view on ADMISSIONS + FEATURE_STORE
- Generic function that calls `READMISSION_PREDICTOR` (the model you registered)
- Natural language interface: *"Which patients admitted last week are high readmission risk?"*